## Bloco 1: KPIs de Cabeçalho e Tendência Temporal (Visão Executiva)

### 1.1 Cartões de KPIs Executivos (Mês Mais Recente)

In [0]:
%sql
SELECT 
    ano_mes_desc                                          AS mes_referencia,
    mvp.gold.fmt_moeda_br(montante_total_reais)           AS montante_total,
    mvp.gold.fmt_pct_br(variacao_montante_mom_pct, 2)     AS variacao_montante_mom,
    mvp.gold.fmt_inteiro_br(total_transacoes)             AS total_transacoes,
    mvp.gold.fmt_pct_br(variacao_transacoes_mom_pct, 2)   AS variacao_transacoes_mom,
    mvp.gold.fmt_moeda_br(ticket_medio_geral_reais)       AS ticket_medio_geral,
    mvp.gold.fmt_pct_br(taxa_contestacao_pct, 4)          AS taxa_contestacao,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)   AS taxa_recuperacao_med
FROM mvp.gold.vw_analise_tendencia_temporal
ORDER BY sk_tempo DESC
LIMIT 1;

mes_referencia,montante_total,variacao_montante_mom,total_transacoes,variacao_transacoes_mom,ticket_medio_geral,taxa_contestacao,taxa_recuperacao_med
2026/08,"R$ 2.484.030.172.514,20","33,87%",5.722.755.497,"119,45%","R$ 434,06","0,0000%","0,00%"


### 1.2 Gráfico de Linha e Eixo Duplo (Volume Financeiro vs. Taxa de Contestação)

In [0]:
%sql
SELECT 
    mvp.gold.fmt_data_br(data_competencia)              AS data_competencia_br,
    ano_mes_desc,
    mvp.gold.fmt_moeda_br(montante_total_reais)          AS montante_total_br,
    mvp.gold.fmt_inteiro_br(total_transacoes)            AS total_transacoes_br,
    mvp.gold.fmt_moeda_br(ticket_medio_geral_reais)      AS ticket_medio_br,
    mvp.gold.fmt_pct_br(taxa_contestacao_pct, 4)         AS taxa_contestacao_br,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)  AS taxa_recuperacao_med_br,
    mvp.gold.fmt_pct_br(variacao_montante_mom_pct, 2)    AS variacao_montante_mom_br
FROM mvp.gold.vw_analise_tendencia_temporal
ORDER BY sk_tempo ASC;

data_competencia_br,ano_mes_desc,montante_total_br,total_transacoes_br,ticket_medio_br,taxa_contestacao_br,taxa_recuperacao_med_br,variacao_montante_mom_br
01/11/2020,2020/11,"R$ 53.890.784,93",71.010,"R$ 758,92","0,0000%","0,00%",null
01/12/2020,2020/12,"R$ 3.293.600.107,40",524.441,"R$ 6.280,21","0,0000%","0,00%","6.011,62%"
01/01/2021,2021/01,"R$ 817.058.283,67",1.802.153,"R$ 453,38","0,0000%","0,00%","-75,19%"
01/02/2021,2021/02,"R$ 361.005.193,43",532.659,"R$ 677,74","0,0000%","0,00%","-55,82%"
01/03/2021,2021/03,"R$ 4.232.818.311,21",6.377.119,"R$ 663,75","0,0000%","0,00%","1.072,51%"
01/04/2021,2021/04,"R$ 443.519.465,59",1.652.651,"R$ 268,37","0,0000%","0,00%","-89,52%"
01/05/2021,2021/05,"R$ 3.478.503.140,57",12.566.816,"R$ 276,80","0,0000%","0,00%","684,30%"
01/06/2021,2021/06,"R$ 1.879.406.936,93",5.304.238,"R$ 354,32","0,0000%","0,00%","-45,97%"
01/07/2021,2021/07,"R$ 4.966.590.578,59",13.970.167,"R$ 355,51","0,0000%","0,00%","164,26%"
01/08/2021,2021/08,"R$ 3.218.501.306,42",8.740.037,"R$ 368,25","0,0000%","0,00%","-35,20%"


# Bloco 2: Análise Regional e Demografia IBGE (Per Capita)

### 2.1 Ranking Regional Consolidado e Intensidade Populacional

In [0]:
%sql
WITH agregacao_regional AS (
    SELECT 
        nome_regiao,
        populacao_recente,
        SUM(volume_transacoes_pagas)                                            AS total_transacoes_num,
        SUM(montante_total_pago_reais)                                          AS montante_total_pago_num,
        ROUND(SUM(montante_total_pago_reais) / NULLIF(SUM(volume_transacoes_pagas), 0), 2) AS ticket_medio_regiao_num,
        ROUND(
            SUM(montante_total_pago_reais) / NULLIF(SUM(SUM(montante_total_pago_reais)) OVER (), 0) * 100, 
            2
        )                                                                       AS share_nacional_pct_num,
        ROUND(SUM(montante_total_pago_reais) / NULLIF(populacao_recente, 0), 2) AS gasto_per_capita_reais_num,
        ROUND(SUM(volume_transacoes_pagas) * 1.0 / NULLIF(populacao_recente, 0), 2) AS transacoes_por_habitante_num
    FROM mvp.gold.vw_analise_ranking_regional
    GROUP BY nome_regiao, populacao_recente
)
SELECT 
    nome_regiao,
    mvp.gold.fmt_inteiro_br(populacao_recente)                             AS populacao,
    mvp.gold.fmt_inteiro_br(total_transacoes_num)                          AS total_transacoes,
    mvp.gold.fmt_moeda_br(montante_total_pago_num)                         AS montante_total_pago,
    mvp.gold.fmt_moeda_br(ticket_medio_regiao_num)                         AS ticket_medio_regiao,
    mvp.gold.fmt_pct_br(share_nacional_pct_num, 2)                         AS share_nacional_pct,
    mvp.gold.fmt_moeda_br(gasto_per_capita_reais_num)                      AS gasto_per_capita,
    TRANSLATE(FORMAT_NUMBER(transacoes_por_habitante_num, 2), ',.', '.,')   AS transacoes_por_habitante
FROM agregacao_regional
ORDER BY gasto_per_capita_reais_num DESC;

nome_regiao,populacao,total_transacoes,montante_total_pago,ticket_medio_regiao,share_nacional_pct,gasto_per_capita,transacoes_por_habitante
SUL,29.937.706,2.853.536.643,"R$ 1.895.555.489.570,43","R$ 664,28","19,86%","R$ 63.316,66","95,32"
SUDESTE,84.840.113,9.811.181.079,"R$ 5.226.572.441.285,83","R$ 532,72","54,75%","R$ 61.604,97","115,64"
CENTRO-OESTE,16.289.538,1.901.178.896,"R$ 791.102.353.453,36","R$ 416,11","8,29%","R$ 48.565,06","116,71"
NORTE,17.354.884,1.742.575.947,"R$ 432.003.261.487,38","R$ 247,91","4,53%","R$ 24.892,32","100,41"
NORDESTE,54.658.515,5.652.005.655,"R$ 1.200.189.434.584,52","R$ 212,35","12,57%","R$ 21.957,96","103,41"


### 2.2 Balança de Fluxos Inter-Regionais (Envio vs. Recebimento)

In [0]:
%sql
WITH envios AS (
    SELECT 
        r.nome_regiao,
        SUM(f.montante_reais) AS montante_enviado
    FROM mvp.gold.fato_pix_transacoes f
    INNER JOIN mvp.gold.dim_regiao r ON f.sk_regiao_pagador = r.sk_regiao
    WHERE r.id_regiao_ibge > 0
    GROUP BY r.nome_regiao
),
recebimentos AS (
    SELECT 
        r.nome_regiao,
        SUM(f.montante_reais) AS montante_recebido
    FROM mvp.gold.fato_pix_transacoes f
    INNER JOIN mvp.gold.dim_regiao r ON f.sk_regiao_recebedor = r.sk_regiao
    WHERE r.id_regiao_ibge > 0
    GROUP BY r.nome_regiao
),
consolidado AS (
    SELECT 
        e.nome_regiao,
        e.montante_enviado,
        rc.montante_recebido,
        ROUND(rc.montante_recebido - e.montante_enviado, 2) AS saldo_liquido_regional,
        CASE 
            WHEN rc.montante_recebido >= e.montante_enviado THEN 'Receptora Líquida'
            ELSE 'Emissora Líquida'
        END AS classificacao_fluxo
    FROM envios e
    INNER JOIN recebimentos rc ON e.nome_regiao = rc.nome_regiao
)
SELECT 
    nome_regiao,
    mvp.gold.fmt_moeda_br(montante_enviado)        AS montante_enviado,
    mvp.gold.fmt_moeda_br(montante_recebido)       AS montante_recebido,
    mvp.gold.fmt_moeda_br(saldo_liquido_regional)  AS saldo_liquido_regional,
    classificacao_fluxo
FROM consolidado
ORDER BY saldo_liquido_regional DESC;

nome_regiao,montante_enviado,montante_recebido,saldo_liquido_regional,classificacao_fluxo
SUDESTE,"R$ 5.226.572.441.285,83","R$ 5.290.874.356.280,83","R$ 64.301.914.995,00",Receptora Líquida
CENTRO-OESTE,"R$ 791.102.353.453,36","R$ 819.057.521.687,39","R$ 27.955.168.234,03",Receptora Líquida
NORTE,"R$ 432.003.261.487,38","R$ 425.339.008.652,86","R$ -6.664.252.834,52",Emissora Líquida
SUL,"R$ 1.895.555.489.570,43","R$ 1.849.965.143.092,21","R$ -45.590.346.478,22",Emissora Líquida
NORDESTE,"R$ 1.200.189.434.584,52","R$ 1.157.202.043.939,01","R$ -42.987.390.645,51",Emissora Líquida


## Bloco 3: Perfil Comportamental, Relações (PF/PJ) e Modalidades

### 3.1 Matriz Relacional de Pagamento (P2P, P2B, B2P, B2B)

In [0]:
%sql
WITH consolidado_fluxo AS (
    SELECT 
        fluxo_relacional_pessoa,
        CASE 
            WHEN fluxo_relacional_pessoa = 'PF ➔ PF' THEN 'P2P (Transferência Pessoal)'
            WHEN fluxo_relacional_pessoa = 'PF ➔ PJ' THEN 'P2B (Varejo / Consumo)'
            WHEN fluxo_relacional_pessoa = 'PJ ➔ PF' THEN 'B2P (Pagamentos / Salários)'
            WHEN fluxo_relacional_pessoa = 'PJ ➔ PJ' THEN 'B2B (Liquidação Corporativa)'
            ELSE 'Outros'
        END                                                                       AS categoria_fluxo,
        SUM(total_operacoes)                                                      AS total_operacoes_num,
        SUM(volume_financeiro_reais)                                              AS volume_financeiro_num,
        ROUND(
            SUM(volume_financeiro_reais) / NULLIF(SUM(SUM(volume_financeiro_reais)) OVER (), 0) * 100, 
            2
        )                                                                         AS share_volume_pct_num,
        ROUND(SUM(volume_financeiro_reais) / NULLIF(SUM(total_operacoes), 0), 2)  AS ticket_medio_num
    FROM mvp.gold.vw_analise_perfil_risco_operacional
    GROUP BY fluxo_relacional_pessoa
)
SELECT 
    fluxo_relacional_pessoa,
    categoria_fluxo,
    mvp.gold.fmt_inteiro_br(total_operacoes_num)      AS total_operacoes,
    mvp.gold.fmt_moeda_br(volume_financeiro_num)      AS volume_financeiro,
    mvp.gold.fmt_pct_br(share_volume_pct_num, 2)      AS share_volume_pct,
    mvp.gold.fmt_moeda_br(ticket_medio_num)           AS ticket_medio
FROM consolidado_fluxo
ORDER BY volume_financeiro_num DESC;

fluxo_relacional_pessoa,categoria_fluxo,total_operacoes,volume_financeiro,share_volume_pct,ticket_medio
PJ ➔ PJ,B2B (Liquidação Corporativa),890.828.298,"R$ 5.088.339.940.153,09","53,20%","R$ 5.711,92"
PF ➔ PF,P2P (Transferência Pessoal),8.643.345.170,"R$ 2.307.979.108.023,21","24,13%","R$ 267,02"
PF ➔ PJ,P2B (Varejo / Consumo),10.527.490.907,"R$ 1.108.118.258.448,46","11,59%","R$ 105,26"
PJ ➔ PF,B2P (Pagamentos / Salários),1.913.944.218,"R$ 1.059.583.706.695,47","11,08%","R$ 553,61"
NAO DISPONIVEL ➔ NAO DISPONIVEL,Outros,880.460,"R$ 23.381.684,76","0,00%","R$ 26,56"


### 3.2 Penetração por Canal de Iniciação e Faixa Etária do Pagador

In [0]:
%sql
WITH metricas_faixa_modalidade AS (
    SELECT 
        faixa_etaria_pagador,
        ordem_faixa_etaria_pagador,
        forma_iniciacao,
        forma_iniciacao_desc,
        ordem_iniciacao,
        SUM(total_operacoes)                                                     AS total_operacoes_num,
        SUM(volume_financeiro_reais)                                             AS volume_financeiro_num,
        ROUND(SUM(volume_financeiro_reais) / NULLIF(SUM(total_operacoes), 0), 2) AS ticket_medio_num
    FROM mvp.gold.vw_analise_perfil_risco_operacional
    GROUP BY 
        faixa_etaria_pagador,
        ordem_faixa_etaria_pagador,
        forma_iniciacao,
        forma_iniciacao_desc,
        ordem_iniciacao
)
SELECT 
    faixa_etaria_pagador,
    forma_iniciacao,
    forma_iniciacao_desc,
    mvp.gold.fmt_inteiro_br(total_operacoes_num)   AS total_operacoes,
    mvp.gold.fmt_moeda_br(volume_financeiro_num)   AS volume_financeiro,
    mvp.gold.fmt_moeda_br(ticket_medio_num)        AS ticket_medio
FROM metricas_faixa_modalidade
ORDER BY 
    ordem_faixa_etaria_pagador ASC,
    volume_financeiro_num DESC;

faixa_etaria_pagador,forma_iniciacao,forma_iniciacao_desc,total_operacoes,volume_financeiro,ticket_medio
entre 20 e 29 anos,DICT,"Chave Pix (Endereçamento via CPF/CNPJ, E-mail, Celular ou EVP)",1.385.834.265,"R$ 327.569.213.732,98","R$ 236,37"
entre 20 e 29 anos,QRDN,QR Code Dinâmico (Cobrança com Valor e Conciliação no Varejo/API),2.350.787.823,"R$ 133.914.437.596,61","R$ 56,97"
entre 20 e 29 anos,MANU,Inserção Manual (Dados de Agência e Conta),499.510.305,"R$ 99.339.444.778,41","R$ 198,87"
entre 20 e 29 anos,QRES,QR Code Estático (Ponto de Venda Físico / P2P com Chave Fixa),404.576.251,"R$ 30.211.988.504,43","R$ 74,68"
entre 20 e 29 anos,NAO DISPONIVEL,Outras Modalidades de Iniciação,21.312.225,"R$ 6.291.491.135,24","R$ 295,21"
entre 20 e 29 anos,INIC,Iniciador de Pagamento (Open Finance / Sem Redirecionamento Manual),2.464.287,"R$ 815.109.469,65","R$ 330,77"
entre 20 e 29 anos,AUTO,Outras Modalidades de Iniciação,1.524.573,"R$ 42.376.871,92","R$ 27,80"
entre 20 e 29 anos,APDN,Outras Modalidades de Iniciação,522.762,"R$ 17.087.194,12","R$ 32,69"
entre 20 e 29 anos,APES,Outras Modalidades de Iniciação,4.951,"R$ 212.613,95","R$ 42,94"
entre 20 e 29 anos,OUTRA,Outras Modalidades de Iniciação,193,"R$ 40.421,45","R$ 209,44"


## Bloco 4: Inteligência de Segurança e Eficácia do MED

### 4.1 Funil de Efetividade do MED

In [0]:
%sql
WITH funil_med AS (
    SELECT 
        m.sk_tempo,
        t.ano_mes_desc,
        m.qtde_pix_contestados,
        m.qtde_contestacoes_aceitas,
        (m.qtde_devolucoes_integrais + m.qtde_devolucoes_parciais)                     AS total_devolucoes_efetuadas,
        ROUND((m.qtde_contestacoes_aceitas * 100.0 / NULLIF(m.qtde_pix_contestados, 0)), 2) AS taxa_aceite_contestacao_pct,
        ROUND(
            ((m.qtde_devolucoes_integrais + m.qtde_devolucoes_parciais) * 100.0 / NULLIF(m.qtde_contestacoes_aceitas, 0)), 
            2
        )                                                                             AS taxa_conclusao_devolucao_pct,
        m.valor_contestacoes_aceitas_reais,
        (m.valor_devolvido_integral_reais + m.valor_devolvido_parcial_reais)           AS valor_total_recuperado_reais,
        m.taxa_efetiva_recuperacao_pct
    FROM mvp.gold.fato_pix_fraudes_med m
    INNER JOIN mvp.gold.dim_tempo t ON m.sk_tempo = t.sk_tempo
)
SELECT 
    ano_mes_desc,
    mvp.gold.fmt_inteiro_br(qtde_pix_contestados)                AS qtde_pix_contestados,
    mvp.gold.fmt_inteiro_br(qtde_contestacoes_aceitas)           AS qtde_contestacoes_aceitas,
    mvp.gold.fmt_inteiro_br(total_devolucoes_efetuadas)          AS total_devolucoes_efetuadas,
    mvp.gold.fmt_pct_br(taxa_aceite_contestacao_pct, 2)          AS taxa_aceite_contestacao_pct,
    mvp.gold.fmt_pct_br(taxa_conclusao_devolucao_pct, 2)         AS taxa_conclusao_devolucao_pct,
    mvp.gold.fmt_moeda_br(valor_contestacoes_aceitas_reais)      AS valor_contestacoes_aceitas,
    mvp.gold.fmt_moeda_br(valor_total_recuperado_reais)          AS valor_total_recuperado,
    mvp.gold.fmt_pct_br(taxa_efetiva_recuperacao_pct, 2)         AS taxa_efetiva_recuperacao_med
FROM funil_med
ORDER BY sk_tempo ASC;

ano_mes_desc,qtde_pix_contestados,qtde_contestacoes_aceitas,total_devolucoes_efetuadas,taxa_aceite_contestacao_pct,taxa_conclusao_devolucao_pct,valor_contestacoes_aceitas,valor_total_recuperado,taxa_efetiva_recuperacao_med
2022/01,71.508,57.777,18.069,"80,80%","31,27%","R$ 122.558.868,28","R$ 6.488.811,31","3,48%"
2022/02,78.636,60.680,18.011,"77,17%","29,68%","R$ 136.034.158,58","R$ 8.148.164,31","3,94%"
2022/03,134.964,105.016,30.318,"77,81%","28,87%","R$ 216.939.933,12","R$ 17.011.303,74","5,07%"
2022/04,152.636,114.763,28.422,"75,19%","24,77%","R$ 178.163.735,34","R$ 12.077.469,08","4,77%"
2022/05,229.401,149.006,33.837,"64,95%","22,71%","R$ 224.862.932,69","R$ 12.336.852,48","3,64%"
2022/06,190.206,146.526,38.583,"77,04%","26,33%","R$ 223.677.609,87","R$ 19.496.400,20","6,72%"
2022/07,186.056,142.493,39.199,"76,59%","27,51%","R$ 213.715.054,18","R$ 14.154.425,38","4,58%"
2022/08,218.131,155.021,43.841,"71,07%","28,28%","R$ 233.907.823,75","R$ 15.637.349,00","4,60%"
2022/09,196.377,137.732,42.648,"70,14%","30,96%","R$ 220.824.321,90","R$ 19.309.199,72","5,81%"
2022/10,199.364,147.113,45.608,"73,79%","31,00%","R$ 232.190.741,27","R$ 22.190.107,99","6,69%"


### 4.2 Diagnóstico dos Gargalos de Devolução (Por que o dinheiro não volta?)

In [0]:
%sql
WITH detalhe_gargalos AS (
    SELECT 
        m.sk_tempo,
        t.ano_mes_desc,
        m.valor_contestacoes_aceitas_reais,
        (m.valor_devolvido_integral_reais + m.valor_devolvido_parcial_reais) AS valor_recuperado,
        m.valor_nao_devolvido_saldo_insuficiente                            AS valor_perdido_saldo_insuficiente,
        m.valor_nao_devolvido_conta_encerrada                               AS valor_perdido_conta_encerrada,
        m.valor_nao_devolvido_motivos_diversos                              AS valor_perdido_outros_motivos,
        ROUND(
            (m.valor_nao_devolvido_saldo_insuficiente * 100.0 / NULLIF(m.valor_contestacoes_aceitas_reais, 0)), 
            2
        ) AS impacto_saldo_insuficiente_pct,
        ROUND(
            (m.valor_nao_devolvido_conta_encerrada * 100.0 / NULLIF(m.valor_contestacoes_aceitas_reais, 0)), 
            2
        ) AS impacto_conta_encerrada_pct
    FROM mvp.gold.fato_pix_fraudes_med m
    INNER JOIN mvp.gold.dim_tempo t ON m.sk_tempo = t.sk_tempo
)
SELECT 
    ano_mes_desc,
    mvp.gold.fmt_moeda_br(valor_contestacoes_aceitas_reais)        AS valor_contestacoes_aceitas,
    mvp.gold.fmt_moeda_br(valor_recuperado)                        AS valor_recuperado,
    mvp.gold.fmt_moeda_br(valor_perdido_saldo_insuficiente)        AS valor_perdido_saldo_insuficiente,
    mvp.gold.fmt_moeda_br(valor_perdido_conta_encerrada)          AS valor_perdido_conta_encerrada,
    mvp.gold.fmt_moeda_br(valor_perdido_outros_motivos)           AS valor_perdido_outros_motivos,
    mvp.gold.fmt_pct_br(impacto_saldo_insuficiente_pct, 2)         AS impacto_saldo_insuficiente_pct,
    mvp.gold.fmt_pct_br(impacto_conta_encerrada_pct, 2)           AS impacto_conta_encerrada_pct
FROM detalhe_gargalos
ORDER BY sk_tempo DESC;

ano_mes_desc,valor_contestacoes_aceitas,valor_recuperado,valor_perdido_saldo_insuficiente,valor_perdido_conta_encerrada,valor_perdido_outros_motivos,impacto_saldo_insuficiente_pct,impacto_conta_encerrada_pct
2026/04,"R$ 598.293.955,83","R$ 97.081.452,97","R$ 309.177.755,99","R$ 12.784.586,62","R$ 29.277.646,60","51,68%","2,14%"
2026/03,"R$ 705.389.899,96","R$ 93.031.656,03","R$ 402.955.730,29","R$ 11.501.938,40","R$ 26.884.953,19","57,13%","1,63%"
2026/02,"R$ 655.822.072,41","R$ 87.282.465,26","R$ 305.146.250,98","R$ 14.335.034,01","R$ 67.570.770,18","46,53%","2,19%"
2026/01,"R$ 723.028.862,55","R$ 111.374.731,48","R$ 355.850.367,79","R$ 12.358.751,79","R$ 53.703.986,64","49,22%","1,71%"
2025/12,"R$ 573.891.428,27","R$ 55.079.975,14","R$ 303.014.583,66","R$ 8.299.089,51","R$ 29.654.498,64","52,80%","1,45%"
2025/11,"R$ 859.860.707,12","R$ 58.828.752,41","R$ 477.280.642,45","R$ 33.350.600,32","R$ 84.174.825,52","55,51%","3,88%"
2025/10,"R$ 769.478.838,19","R$ 70.616.962,10","R$ 406.205.136,89","R$ 8.753.124,76","R$ 52.870.838,67","52,79%","1,14%"
2025/09,"R$ 610.598.393,15","R$ 51.014.648,56","R$ 317.597.358,20","R$ 9.160.789,52","R$ 34.545.621,92","52,01%","1,50%"
2025/08,"R$ 727.671.388,95","R$ 88.061.915,83","R$ 311.768.691,54","R$ 13.935.492,94","R$ 75.992.425,05","42,84%","1,92%"
2025/07,"R$ 698.566.565,98","R$ 69.311.353,15","R$ 345.866.356,42","R$ 12.980.548,19","R$ 53.022.123,99","49,51%","1,86%"
